In [25]:

import pandas as pd 
from openai import OpenAI
from dotenv import load_dotenv
load_dotenv()

client = OpenAI(
    api_key="sk-yTJxTqRrqRdP0DKDqwPxDyHHE149jcis5x4qFwOUR9mIaP6D",
    base_url="https://api.chatanywhere.tech/v1"
)

In [2]:
failures = pd.read_csv(r"../data/csv/failures_updated.csv")   
df = pd.read_excel("../builder/classified.xlsx") 

In [3]:
failures

,model_name,failure_profile,start_time,end_time,failure_interval,failure_type,solution_method,is_fixable
0,*.Models.Model.Assembly3,FailureA,8:56:00.3331,9:12:02.0050,1:04.4022,1,5,0
1,*.Models.Model.Assembly3,FailureA,9:13:06.4072,9:51:19.4638,2:09:30.8702,1,3,0
2,*.Models.Model.Assembly1,FailureA,10:03:51.8516,10:25:35.8209,1:00:00.0000,1,1,0
3,*.Models.Model.Assembly4,FailureA,10:28:16.7044,10:54:18.0976,14:53.4160,2,3,0
4,*.Models.Model.Assembly4,FailureA,11:09:19.6492,11:34:52.1108,8:39.6404,2,2,0
...,...,...,...,...,...,...,...,...
2311,*.Models.Model.Assembly3,FailureA,58:15:47:36.9327,58:16:21:20.3627,1:34:36.1020,1,4,0
2312,*.Models.Model.Assembly1,FailureA,58:15:40:58.0519,58:16:45:44.5882,1:00:00.0000,2,3,0
2313,*.Models.Model.Assembly4,FailureA,58:16:02:35.6456,58:17:01:32.0588,51:03.7902,1,3,0
2314,*.Models.Model.Assembly1,FailureA,58:16:46:05.9819,58:17:44:51.1963,1:00:00.0000,1,6,1


In [4]:
#failures type1 选每个分类的第一行的第一个
#failures type2 选每个分类的第一行的第2个

In [5]:
category_list = list(set(df['category']))
solution = []
for ind,row in failures.iterrows():
    category = category_list[int(row['solution_method'])-1]
    tmp = df[df['category'] == category]
    tmp.index = range(len(tmp))
    solution.append(
        eval(tmp.loc[0,'solutions'])[int(row['failure_type'])])

In [6]:
failures['solution'] = solution

In [7]:
failures.columns

Index(['model_name', 'failure_profile', 'start_time', 'end_time',
       'failure_interval', 'failure_type', 'solution_method', 'is_fixable',
       'solution'],
      dtype='object')

In [12]:
def generate_operator(content,solution):
    response = client.chat.completions.create(
        model="deepseek-reasoner",
        messages=[{"role":"system","content":"""我会给你一个机器运行记录和要操作的内容，你需要把它改写成operator输出的口吻
    Examples：
    1.Sensor disconnection was reported between 43:27 and 1:27:14, requiring manual intervention for resolution. The machine is currently operational, but predicted to fail again in approximately 59 minutes."
    2.The sensor disconnected at 5:41:41 and remained offline for 19:12:16, requiring manual intervention to resolve. The machine is currently operational, but it is predicted to experience this issue again within the next hour.
    3.The most important issue in this failure record is that the machine experienced an actuator failure, which is a critical component responsible for controlling the machine's movement and operation. The fact that the failure required manual intervention for repair and has a predicted time until next failure of only 8 hours and 21 minutes indicates a high likelihood of recurrence, posing a significant risk to machine
    #Ouput:
    {"operator":"具体改写的内容"}
    不需要返回其他多余解释
    
    #NOTE:
    1.Reply in English.
         """
                   },
    {"role":"user","content":f"这是机器运行记录：{content}\n和要操作的内容：{solution}"}],
        temperature=0.01
    )
    return response.choices[0].message.content

In [ ]:
results = []

In [26]:
for i,row in failures[1853:].iterrows():
    print(i)
    content = ""
    for c in failures.columns[:-1]:
        content += f"{c}:{row[c]}"
    solution = row['solution']
    res = generate_operator(content,solution)
    results.append(res)

1853
1854
1855
1856
1857
1858
1859
1860
1861
1862
1863
1864
1865
1866
1867
1868
1869
1870
1871
1872
1873
1874
1875
1876
1877
1878
1879
1880
1881
1882
1883
1884
1885
1886
1887
1888
1889
1890
1891
1892
1893
1894
1895
1896
1897
1898
1899
1900
1901
1902
1903
1904
1905
1906
1907
1908
1909
1910
1911
1912
1913
1914
1915
1916
1917
1918
1919
1920
1921
1922
1923
1924
1925
1926
1927
1928
1929
1930
1931
1932
1933
1934
1935
1936
1937
1938
1939
1940
1941
1942
1943
1944
1945
1946
1947
1948
1949
1950
1951
1952
1953
1954
1955
1956
1957
1958
1959
1960
1961
1962
1963
1964
1965
1966
1967
1968
1969
1970
1971
1972
1973
1974
1975
1976
1977
1978
1979
1980
1981
1982
1983
1984
1985
1986
1987
1988
1989
1990
1991
1992
1993
1994
1995
1996
1997
1998
1999
2000
2001
2002
2003
2004
2005
2006
2007
2008
2009
2010
2011
2012
2013
2014
2015
2016
2017
2018
2019
2020
2021
2022
2023
2024
2025
2026
2027
2028
2029
2030
2031
2032
2033
2034
2035
2036
2037
2038
2039
2040
2041
2042
2043
2044
2045
2046
2047
2048
2049
2050
2051
2052


In [28]:
results 

['{"operator":"The machine experienced a failure starting at 8:56:00 and ending at 9:12:02, with a failure interval of 1 minute and 4 seconds. The failure type is categorized as type 1, and the solution method used was method 5. However, the issue is not fixable. Please verify the voltage at the I/O PCB and the motor to ensure proper operation."}',
 '{"operator":"The machine experienced a failure starting at 9:13:06 and ending at 9:51:19, with a failure interval of 2 hours, 9 minutes, and 30 seconds. The failure type was identified as type 1, and the solution method used was method 3. Unfortunately, the issue is not fixable at this time. Please check the voltage to the turret clamp or unclamp solenoid as part of the troubleshooting process."}',
 '{"operator":"The machine experienced a failure starting at 10:03:51 and ending at 10:25:35, with a failure interval of 1 hour. The failure type is classified as type 1, and the solution method is also type 1. Unfortunately, this issue is not f

In [33]:
failures['operator_solution'] =[eval(i)['operator'] for i in results]

In [36]:
failures .to_excel("../builder/repair_feedback.xlsx")

In [37]:
failures 

,model_name,failure_profile,start_time,end_time,failure_interval,failure_type,solution_method,is_fixable,solution,operator_solution
0,*.Models.Model.Assembly3,FailureA,8:56:00.3331,9:12:02.0050,1:04.4022,1,5,0,Verify the voltage at the I/O PCB and the motor.,The machine experienced a failure starting at ...
1,*.Models.Model.Assembly3,FailureA,9:13:06.4072,9:51:19.4638,2:09:30.8702,1,3,0,Check the voltage to the turret clamp or uncla...,The machine experienced a failure starting at ...
2,*.Models.Model.Assembly1,FailureA,10:03:51.8516,10:25:35.8209,1:00:00.0000,1,1,0,Inspect the gripper airlines and fittings for ...,The machine experienced a failure starting at ...
3,*.Models.Model.Assembly4,FailureA,10:28:16.7044,10:54:18.0976,14:53.4160,2,3,0,Check the turret clamp/unclamp proximity sensor.,The machine experienced a failure starting at ...
4,*.Models.Model.Assembly4,FailureA,11:09:19.6492,11:34:52.1108,8:39.6404,2,2,0,Check the coolant collector for blockage,The machine experienced a failure starting at ...
...,...,...,...,...,...,...,...,...,...,...
2311,*.Models.Model.Assembly3,FailureA,58:15:47:36.9327,58:16:21:20.3627,1:34:36.1020,1,4,0,Check the switch and key alignment,The machine experienced a failure starting at ...
2312,*.Models.Model.Assembly1,FailureA,58:15:40:58.0519,58:16:45:44.5882,1:00:00.0000,2,3,0,Check the turret clamp/unclamp proximity sensor.,"The machine experienced a failure of type 2, s..."
2313,*.Models.Model.Assembly4,FailureA,58:16:02:35.6456,58:17:01:32.0588,51:03.7902,1,3,0,Check the voltage to the turret clamp or uncla...,Please check the voltage to the turret clamp o...
2314,*.Models.Model.Assembly1,FailureA,58:16:46:05.9819,58:17:44:51.1963,1:00:00.0000,1,6,1,set the maximum coolant from remaining concent...,The machine experienced a failure starting at ...
